# L3 — Order Agent

## What You'll Learn

- How to create a Lambda-backed tool and expose it through the AgentCore Gateway with an MCP schema
- How to deploy a Strands agent as an A2A server on AgentCore Runtime
- How to enable CloudWatch log delivery and tracing for a deployed agent
- How to register an agent in the AgentCore Registry so the Harness can discover it

## Overview

The Order Agent handles all order-related questions from AnyCompany customers — looking up a specific order by ID or retrieving all orders for a customer. This notebook builds the agent end-to-end: a Lambda function that queries DynamoDB, a Gateway Target that exposes it as an MCP tool, a Strands A2A agent deployed on AgentCore Runtime, and a Registry record that makes it discoverable by the Harness.

## Architecture

```
  Harness (Orchestrator)
       │
       │  discovers via AgentCore Registry
       ▼
  Order Agent (Strands, A2A on AgentCore Runtime)
       │
       │  calls tool via AgentCore Gateway (MCP)
       ▼
  Lambda: anycompany_order_tools
       │
       ├── order_id   ──► DynamoDB OrderIdIndex GSI
       └── customer_id──► DynamoDB CustomerOrders (partition key)
```

## Steps

1. Create the Lambda execution role and deploy the order lookup Lambda
2. Create the Gateway Target (MCP tool schema)
3. Deploy the Order Agent to AgentCore Runtime
4. Test the agent via the Runtime endpoint
5. Register the agent in the AgentCore Registry
6. Save resource IDs to SSM

## License Details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- `1_pre-requisites.ipynb` completed — Registry, Gateway, and Cognito IDs must be in SSM
- `1_pluggable_inference_layer.ipynb` completed — `model_id` must be in SSM
- `2_dynamodb_tables.ipynb` completed — `dynamodb_orders_table` must be in SSM
- `boto3`, `strands-agents`, and `bedrock_agentcore_starter_toolkit` (installed in the first code cell)

Install required packages.

In [ ]:
%pip install --quiet boto3==1.43.0 strands-agents==1.43.0 strands-agents-tools==0.2.0 bedrock-agentcore==1.14.0 bedrock-agentcore-starter-toolkit==0.3.6

Import libraries, set the region, and load shared configuration from SSM.

In [ ]:
import boto3
import json
import time
import os
import io
import zipfile
import uuid
from datetime import datetime, timedelta

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1") # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"

ssm = boto3.client("ssm", region_name=REGION)
iam = boto3.client("iam")
sts = boto3.client("sts", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)

def get_ssm(name, default=None):
    try:
        return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]
    except ssm.exceptions.ParameterNotFound:
        if default is not None:
            return default
        raise

ACCOUNT_ID = sts.get_caller_identity()["Account"]
REGISTRY_ID = get_ssm(f"{SSM_PREFIX}/registry_id")
GATEWAY_ID = get_ssm(f"{SSM_PREFIX}/gateway_id")
GATEWAY_ARN = get_ssm(f"{SSM_PREFIX}/gateway_arn")
GATEWAY_URL = get_ssm(f"{SSM_PREFIX}/gateway_url")
MODEL_ID = get_ssm(f"{SSM_PREFIX}/model_id") #This is going to come from l2 layer.
#MODEL_ID = "us.anthropic.claude-sonnet-4-6" #delete this once l2 layer is tested.
ORDERS_TABLE = get_ssm(f"{SSM_PREFIX}/dynamodb_orders_table")

print(f"Account:       {ACCOUNT_ID}")
print(f"Region:        {REGION}")
print(f"Registry:      {REGISTRY_ID}")
print(f"Gateway:       {GATEWAY_ID}")
print(f"Gateway URL:   {GATEWAY_URL}")
print(f"Model:         {MODEL_ID}")
print(f"Orders Table:  {ORDERS_TABLE}")


## Step 1: Create Lambda Function for Order Lookup Tool

Create the IAM execution role for the Lambda with permissions for CloudWatch Logs, DynamoDB, and SSM.

In [ ]:
LAMBDA_ROLE_NAME = "AnyCompanyOrderToolsLambdaRole"

lambda_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
})

lambda_perms = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany_order_tools"},
        {"Effect": "Allow", "Action": ["logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/lambda/anycompany_order_tools:*"},
        {"Effect": "Allow", "Action": ["dynamodb:GetItem", "dynamodb:Query", "dynamodb:Scan"], "Resource": [f"arn:aws:dynamodb:*:{ACCOUNT_ID}:table/CustomerOrders", f"arn:aws:dynamodb:*:{ACCOUNT_ID}:table/CustomerOrders/index/*"]},
        {"Effect": "Allow", "Action": ["ssm:GetParameter"], "Resource": f"arn:aws:ssm:{REGION}:*:parameter{SSM_PREFIX}/*"}
    ]
})

try:
    r = iam.create_role(RoleName=LAMBDA_ROLE_NAME, AssumeRolePolicyDocument=lambda_trust)
    LAMBDA_ROLE_ARN = r["Role"]["Arn"]
    iam.put_role_policy(RoleName=LAMBDA_ROLE_NAME, PolicyName="OrderToolsPerms", PolicyDocument=lambda_perms)
    print(f"Created: {LAMBDA_ROLE_ARN}")
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{LAMBDA_ROLE_NAME}"
    print(f"Exists: {LAMBDA_ROLE_ARN}")

Package the Lambda handler code as a zip and deploy `anycompany_order_tools` (Python 3.12). Grants the Gateway permission to invoke it.

In [ ]:
LAMBDA_CODE = """import json, os, boto3, logging
from decimal import Decimal

logger = logging.getLogger()
logger.setLevel(logging.INFO)

REGION = os.environ.get("AGENT_REGION", "us-east-1")
ORDERS_TABLE = os.environ.get("ORDERS_TABLE", "CustomerOrders")

dynamodb = boto3.resource("dynamodb", region_name=REGION)
orders_table = dynamodb.Table(ORDERS_TABLE)


class DecimalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)
        return super().default(obj)


def _serialize(data):
    return json.loads(json.dumps(data, cls=DecimalEncoder))


def lambda_handler(event, context):
    logger.info(f"Processing request - order_id present: {bool(event.get('order_id'))}, customer_id present: {bool(event.get('customer_id'))}")

    order_id = event.get("order_id", "")
    customer_id = event.get("customer_id", "")

    try:
        # Case 1: Lookup by order_id (uses GSI OrderIdIndex)
        if order_id:
            from boto3.dynamodb.conditions import Key
            resp = orders_table.query(
                IndexName="OrderIdIndex",
                KeyConditionExpression=Key("orderId").eq(order_id),
            )
            items = resp.get("Items", [])
            if not items:
                result = {"status": "not_found", "message": f"No order found with ID {order_id}"}
            else:
                result = {"status": "success", "order": _serialize(items[0])}

        # Case 2: Lookup by customer_id (partition key)
        elif customer_id:
            from boto3.dynamodb.conditions import Key
            resp = orders_table.query(
                KeyConditionExpression=Key("customerId").eq(customer_id),
            )
            items = resp.get("Items", [])
            if not items:
                result = {"status": "not_found", "message": f"No orders found for customer {customer_id}"}
            else:
                result = {
                    "status": "success",
                    "customer_id": customer_id,
                    "order_count": len(items),
                    "orders": _serialize(items),
                }

        else:
            result = {"status": "error", "message": "Provide at least one of: order_id or customer_id."}

    except Exception as e:
        logger.error(f"Error: {e}")
        result = {"status": "error", "message": str(e)}

    logger.info(f"Request completed - status: {result.get('status')}, record_count: {len(result.get('orders', [result.get('order')] if result.get('order') else []))}")
    return result
"""

# Package as zip
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', LAMBDA_CODE)
zip_bytes = buf.getvalue()

LAMBDA_NAME = "anycompany_order_tools"

try:
    resp = lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=LAMBDA_ROLE_ARN,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
        Environment={"Variables": {
            "ORDERS_TABLE": ORDERS_TABLE,
            "AGENT_REGION": REGION,
        }},
    )
    LAMBDA_ARN = resp["FunctionArn"]
    print(f"Created: {LAMBDA_ARN}")
except lambda_client.exceptions.ResourceConflictException:
    LAMBDA_ARN = lambda_client.get_function(FunctionName=LAMBDA_NAME)["Configuration"]["FunctionArn"]
    lambda_client.update_function_code(FunctionName=LAMBDA_NAME, ZipFile=zip_bytes)
    # Wait for code update to settle, then refresh env vars so config changes are picked up.
    lambda_client.get_waiter('function_updated_v2').wait(FunctionName=LAMBDA_NAME)
    lambda_client.update_function_configuration(
        FunctionName=LAMBDA_NAME,
        Environment={"Variables": {
            "ORDERS_TABLE": ORDERS_TABLE,
            "AGENT_REGION": REGION,
        }},
    )
    print(f"Updated: {LAMBDA_ARN}")

# Allow Gateway to invoke
try:
    lambda_client.add_permission(
        FunctionName=LAMBDA_NAME,
        StatementId="AllowGatewayInvoke",
        Action="lambda:InvokeFunction",
        Principal="bedrock-agentcore.amazonaws.com",
    )
    print("Added Gateway invoke permission.")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists.")

## Step 2: Create Gateway Target

Register the Lambda as an MCP tool target on the Gateway. Waits for Lambda to be Active, then waits for the target to reach READY.

In [ ]:
TOOL_SCHEMAS = [
    {
        "name": "check_order_details",
        "description": "Look up order details or customer orders from the database. Returns order ID, status, product name, order amount, and customer tier. Provide at least one parameter: order_id for a specific order or customer_id for all orders of a customer.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order identifier (e.g. ORD-10001). Returns details for that specific order."},
                "customer_id": {"type": "string", "description": "The customer identifier (e.g. CUST-789). Returns all orders for that customer."}
            }
        }
    }
]

TARGET_NAME = "order-tools"

# Wait for Lambda to be fully active before creating the Gateway Target
print("Waiting for Lambda to be active...")
for _ in range(20):
    fn = lambda_client.get_function_configuration(FunctionName=LAMBDA_NAME)
    state = fn.get("State", "")
    last_update = fn.get("LastUpdateStatus", "Successful")
    if state == "Active" and last_update in ("Successful", None):
        print(f"Lambda is Active and ready.")
        break
    print(f"  State: {state}, LastUpdateStatus: {last_update} — waiting 5s...")
    time.sleep(5)
else:
    raise TimeoutError("Lambda did not become active within 100s.")

try:
    target_resp = agentcore_control.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name=TARGET_NAME,
        description="Order lookup tools (order details, customer orders) backed by Lambda + DynamoDB",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": LAMBDA_ARN,
                    "toolSchema": {"inlinePayload": TOOL_SCHEMAS}
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    TARGET_ID = target_resp["targetId"]
    print(f"Created Gateway Target: {TARGET_ID}")
except agentcore_control.exceptions.ConflictException:
    targets = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
    TARGET_ID = None
    for t in targets.get("targets", targets.get("items", [])):
        if t.get("name") == TARGET_NAME:
            TARGET_ID = t["targetId"]
            print(f"Already exists: {TARGET_ID}")
            break
    if not TARGET_ID:
        # Try getting by name directly
        t = agentcore_control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_NAME)
        TARGET_ID = t["targetId"]
        print(f"Found by name: {TARGET_ID}")

# Wait for ready
print("Waiting for target...")
for _ in range(20):
    t = agentcore_control.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
    status = t["status"]
    if status == "READY":
        print(f"Gateway Target READY")
        break
    print(f"  {status}...")
    time.sleep(5)

print(f"\nTool available via Gateway: check_order_details")

## Step 3: Deploy to AgentCore Runtime

Create the Runtime IAM role, configure the agent with `order_agent_a2a.py` as the entrypoint, and launch it as an A2A server.

In [ ]:
# Remove cached config to avoid cross-notebook conflicts
import os
if os.path.exists(".bedrock_agentcore.yaml"):
    os.remove(".bedrock_agentcore.yaml")
    print("Cleared cached .bedrock_agentcore.yaml")
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")
    print("Removed existing Dockerfile (will be auto-generated)")

from bedrock_agentcore_starter_toolkit import Runtime

# IAM role for AgentCore Runtime
RUNTIME_ROLE_NAME = "AgentCoreOrderAgentA2ARole"

runtime_trust = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})

try:
    r = iam.create_role(RoleName=RUNTIME_ROLE_NAME, AssumeRolePolicyDocument=runtime_trust, Description="Runtime role for Order A2A Agent")
    RUNTIME_ROLE_ARN = r["Role"]["Arn"]
    print(f"Created: {RUNTIME_ROLE_ARN}")
except iam.exceptions.EntityAlreadyExistsException:
    RUNTIME_ROLE_ARN = iam.get_role(RoleName=RUNTIME_ROLE_NAME)["Role"]["Arn"]
    print(f"Exists: {RUNTIME_ROLE_ARN}")

# Least-privilege runtime execution role policy.
# Based on AWS-documented runtime permissions:
# https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html
# X-Ray tracing requires Resource:* for PutTraceSegments/PutTelemetryRecords per AWS documentation
# CloudWatch metrics require Resource:* but namespace condition limits scope to bedrock-agentcore
iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="OrderAgentRuntimeAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Sid": "BedrockInference", "Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": ["arn:aws:bedrock:*::foundation-model/*", "arn:aws:bedrock:*:*:inference-profile/*"]},
            {"Sid": "ECRPublicAuth", "Effect": "Allow", "Action": ["ecr-public:GetAuthorizationToken", "sts:GetServiceBearerToken"], "Resource": "*"},
            {"Sid": "ECRAuth", "Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
            {"Sid": "ECRAgentImage", "Effect": "Allow", "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"], "Resource": "arn:aws:ecr:*:*:repository/bedrock-agentcore-*"},
            {"Sid": "XRayTracing", "Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords", "xray:GetSamplingRules", "xray:GetSamplingTargets"], "Resource": "*"},
            {"Sid": "CloudWatchMetrics", "Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*", "Condition": {"StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}}},
            {"Sid": "CloudWatchLogsGroup", "Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents", "logs:DescribeLogStreams"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtimes/*"},
            {"Sid": "CloudWatchLogsDescribeGroups", "Effect": "Allow", "Action": ["logs:DescribeLogGroups"], "Resource": f"arn:aws:logs:*:{ACCOUNT_ID}:log-group:*"},
            {"Sid": "AgentCoreWorkloadIdentity", "Effect": "Allow", "Action": ["bedrock-agentcore:GetWorkloadAccessToken", "bedrock-agentcore:GetWorkloadAccessTokenForJWT"], "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:workload-identity-directory/default/workload-identity/*"]},
            {"Sid": "SSMConfig", "Effect": "Allow", "Action": ["ssm:GetParameter"], "Resource": f"arn:aws:ssm:*:*:parameter{SSM_PREFIX}/*"},
        ]
    })
)
print(f"Runtime role ready: {RUNTIME_ROLE_ARN}")
time.sleep(10)

# Configure & Launch
agentcore_rt = Runtime()

configure_response = agentcore_rt.configure(
    entrypoint="agents/order_agent_a2a.py",
    execution_role=RUNTIME_ROLE_ARN,
    auto_create_ecr=True,
    requirements_file="agents/requirements_order_a2a.txt",
    region=REGION,
    agent_name="order_agent_a2a",
    protocol="A2A",
)
print(f"Configured: {configure_response}")

Launch the agent and poll until deployment reaches ACTIVE or READY.

In [ ]:
# Launch the agent (auto_update_on_conflict updates if it already exists)
launch_result = agentcore_rt.launch(auto_update_on_conflict=True)
ORDER_AGENT_ARN = launch_result.agent_arn
print(f"Launched: {ORDER_AGENT_ARN}")

# Wait for deployment
print("Waiting for deployment...", end="")
for _ in range(60):
    status_response = agentcore_rt.status()
    deploy_status = status_response.endpoint["status"]
    if deploy_status in ["ACTIVE", "READY", "FAILED"]:
        break
    print(".", end="", flush=True)
    time.sleep(10)
print(f"\nDeployment status: {deploy_status}")
if deploy_status == "FAILED":
    print("Deployment failed. Check AgentCore console.")

Configure CloudWatch log deliveries (APPLICATION_LOGS, USAGE_LOGS) and tracing for the deployed runtime.

In [ ]:
logs_client = boto3.client("logs", region_name=REGION)

AGENT_LOG_PREFIX = "/aws/vendedlogs/bedrock-agentcore/runtime"
agent_runtime_id = ORDER_AGENT_ARN.split("/")[-1]

# Enable APPLICATION_LOGS delivery
for log_type in ["APPLICATION_LOGS", "USAGE_LOGS"]:
    source_name = f"rt-{agent_runtime_id[:20]}-{log_type.lower()}"
    dest_log_group = f"{AGENT_LOG_PREFIX}/{log_type}/{agent_runtime_id}"
    dest_name = f"rt-{agent_runtime_id[:20]}-{log_type.lower()}-dest"

    # Check if source already exists
    existing_source = None
    sources = logs_client.describe_delivery_sources().get('deliverySources', [])
    for s in sources:
        if ORDER_AGENT_ARN in s.get('resourceArns', []) and s.get('logType') == log_type:
            existing_source = s['name']
            break

    if existing_source:
        source_name = existing_source
        print(f"✓ {log_type} source already exists: {source_name}")
    else:
        try:
            logs_client.put_delivery_source(name=source_name, resourceArn=ORDER_AGENT_ARN, logType=log_type)
            print(f"✓ Created {log_type} source: {source_name}")
        except Exception as e:
            print(f"✗ {log_type} source: {e}")

    # Create log group
    try:
        logs_client.create_log_group(logGroupName=dest_log_group)
    except logs_client.exceptions.ResourceAlreadyExistsException:
        pass

    # Create destination
    try:
        logs_client.put_delivery_destination(
            name=dest_name,
            deliveryDestinationConfiguration={
                "destinationResourceArn": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{dest_log_group}"
            },
        )
    except logs_client.exceptions.ConflictException:
        pass

    # Create delivery
    try:
        logs_client.create_delivery(
            deliverySourceName=source_name,
            deliveryDestinationArn=f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:delivery-destination:{dest_name}",
        )
        print(f"✓ {log_type} delivery created")
    except logs_client.exceptions.ConflictException:
        print(f"✓ {log_type} delivery already exists")
    except Exception as e:
        print(f"✗ {log_type} delivery: {e}")

# Enable Tracing
trace_source_name = f"rt-{agent_runtime_id[:20]}-traces"
try:
    existing_trace = None
    for s in sources:
        if ORDER_AGENT_ARN in s.get('resourceArns', []) and s.get('logType') == 'TRACES':
            existing_trace = s['name']
            break
    if existing_trace:
        print(f"✓ Tracing already enabled")
    else:
        logs_client.put_delivery_source(name=trace_source_name, resourceArn=ORDER_AGENT_ARN, logType='TRACES')
        print(f"✓ Tracing enabled")
except Exception as e:
    print(f"✗ Tracing: {e}")

print(f"\nAgent observability configured for {agent_runtime_id}")

## Step 4: Test the Agent

Invoke the deployed agent via the Runtime endpoint using a Cognito test token.

Authenticate as `gold_customer`, send an A2A JSON-RPC order lookup request, and print the response.

In [ ]:
import requests
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# Resolve ORDER_AGENT_ARN if not already set
if 'ORDER_AGENT_ARN' not in dir() or ORDER_AGENT_ARN is None:
    agentcore_runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)
    runtimes = agentcore_runtime_client.list_agent_runtimes()
    ORDER_AGENT_ARN = None
    for rt in runtimes.get("agentRuntimes", runtimes.get("runtimes", [])):
        if "order_agent_a2a" in rt.get("agentRuntimeArn", rt.get("arn", "")):
            ORDER_AGENT_ARN = rt.get("agentRuntimeArn", rt.get("arn"))
            break
    if not ORDER_AGENT_ARN:
        raise RuntimeError("Could not find order_agent_a2a runtime. Please run Step 4b first.")
    print(f"Resolved agent ARN: {ORDER_AGENT_ARN}")

# Get a Cognito token for the test user (simulates orchestrator passing user token)
cognito_client = boto3.client("cognito-idp", region_name=REGION)
# Retrieve the workshop test password from SSM (stored as SecureString).
# get_ssm() passes WithDecryption=True so we get the plaintext back.
USER_PASSWORD = get_ssm(f"{SSM_PREFIX}/user_password")
auth_resp = cognito_client.initiate_auth(
    ClientId=get_ssm(f"{SSM_PREFIX}/cognito_client_id"),
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "gold_customer", "PASSWORD": USER_PASSWORD},
)
test_token = auth_resp["AuthenticationResult"]["IdToken"]
print(f"Got Cognito token for gold_customer (test user)")

# Build the invocation URL
escaped_arn = quote(ORDER_AGENT_ARN, safe="")
invoke_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations"

# Prepare A2A JSON-RPC request (message/send)
session_id = str(uuid.uuid4())
a2a_payload = json.dumps({
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid.uuid4()),
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "What is the status of order ORD-10001?"}],
            "messageId": str(uuid.uuid4())
        }
    }
})

# Sign request with SigV4 (Runtime inbound) + pass test user token
session = boto3.Session(region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()
headers = {
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    "Authorization": f"Bearer {test_token}",
}
aws_req = AWSRequest(method="POST", url=invoke_url, data=a2a_payload, headers=headers)
SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)

# Invoke the agent
print(f"Invoking Order Agent via Runtime endpoint...")
print(f"  Query: What is the status of order ORD-10001?")
print()

response = requests.post(invoke_url, data=a2a_payload, headers=dict(aws_req.headers), timeout=120)
print(f"Response status: {response.status_code}")

if response.status_code == 200:
    resp_text = response.text
    try:
        resp_json = json.loads(resp_text)
        if "error" in resp_json:
            print(f"\n❌ Error: {json.dumps(resp_json['error'], indent=2)}")
        elif "result" in resp_json:
            result = resp_json["result"]
            artifacts = result.get("artifacts", [])
            for artifact in artifacts:
                for part in artifact.get("parts", []):
                    if part.get("kind") == "text":
                        print(part["text"][:2000])
            print("\n✅ Agent invocation via Runtime endpoint successful!")
        else:
            print(json.dumps(resp_json, indent=2)[:2000])
    except json.JSONDecodeError:
        lines = [l for l in resp_text.strip().split("\n") if l.strip()]
        for line in lines:
            try:
                event = json.loads(line)
                if event.get("result", {}).get("status", {}).get("state") == "completed":
                    for artifact in event.get("result", {}).get("artifacts", []):
                        for part in artifact.get("parts", []):
                            if part.get("kind") == "text":
                                print(part["text"][:2000])
            except json.JSONDecodeError:
                pass
        print("\n✅ Agent invocation via Runtime endpoint successful!")
else:
    print(f"\n❌ Error: {response.status_code}")
    print(response.text[:1000])

## Step 5: Register in the AgentCore Registry

Fetch the A2A agent card from the deployed Runtime endpoint.

In [ ]:
from urllib.parse import quote
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

print(ORDER_AGENT_ARN)
# Fetch the Agent Card from the deployed endpoint
escaped_arn = quote(ORDER_AGENT_ARN, safe="")
card_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{escaped_arn}/invocations/.well-known/agent-card.json"

session = boto3.Session(region_name=REGION)
creds = session.get_credentials().get_frozen_credentials()
aws_req = AWSRequest(method="GET", url=card_url, headers={"X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": str(uuid.uuid4())})
SigV4Auth(creds, "bedrock-agentcore", REGION).add_auth(aws_req)

card_resp = requests.get(card_url, headers=dict(aws_req.headers))
card_resp.raise_for_status()
agent_card = card_resp.json()
print(f"Fetched Agent Card: {agent_card.get('name', 'unknown')}")
print(json.dumps(agent_card, indent=2)[:500])

Create the registry record with the inline agent card and submit it for approval (auto-approved).

In [ ]:
# Register in the Registry with inline agent card
RECORD_NAME = "order_agent_a2a_record"

try:
    reg_resp = agentcore_control.create_registry_record(
        registryId=REGISTRY_ID,
        name=RECORD_NAME,
        description="Order Agent — handles order lookups and customer order history.",
        descriptorType="A2A",
        recordVersion="1.0",
        descriptors={"a2a": {"agentCard": {"inlineContent": json.dumps(agent_card)}}},
    )
    RECORD_ID = reg_resp["recordArn"].rsplit("/", 1)[-1]
    print(f"Created registry record: {RECORD_ID}")
except agentcore_control.exceptions.ConflictException:
    print(f"Record '{RECORD_NAME}' already exists.")
    existing = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
    for rec in existing.get("registryRecords", []):
        if rec.get("name") == RECORD_NAME:
            RECORD_ID = rec["recordId"]
            print(f"  Found: {RECORD_ID}")
            break

# Wait for record to be ready
for _ in range(20):
    rec = agentcore_control.get_registry_record(registryId=REGISTRY_ID, recordId=RECORD_ID)
    if rec["status"] not in ["CREATING", "UPDATING"]:
        break
    time.sleep(2)

# Submit for approval (auto-approved)
agentcore_control.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=RECORD_ID)
print(f"Record submitted for approval. Status: {rec['status']}")

## Step 6: Save Resource IDs to SSM

Publish the Lambda ARN and Gateway Target ID to SSM for use by L4 and L5.

In [ ]:
params = {
    f"{SSM_PREFIX}/order_lambda_arn": LAMBDA_ARN,
    f"{SSM_PREFIX}/order_gateway_target_id": TARGET_ID,
}
for name, value in params.items():
    ssm.put_parameter(Name=name, Value=value, Type="SecureString", Overwrite=True)
    print(f"  {name} = {value}")
print("\nDone.")

## Summary

| Resource | Name | Purpose |
|----------|------|---------|
| Lambda | `anycompany_order_tools` | Handles `check_order_details` — order and customer lookups from DynamoDB |
| Gateway Target | `order-tools` | Exposes the Lambda as an MCP tool via the Gateway |
| AgentCore Runtime | `order_agent_a2a` | Strands A2A agent that calls the Gateway tool |
| Registry Record | `order_agent_a2a_record` | Makes the agent discoverable by the Harness |

---
## Cleanup

Run the following cell to delete all resources created in this notebook and avoid ongoing charges.

**Resources deleted:**
- AgentCore: runtime, registry record, gateway target
- Compute: Lambda function, ECR repository, CodeBuild project
- IAM: Lambda execution role and Runtime execution role
- Observability: CloudWatch log delivery components and log groups
- Configuration: SSM parameters

> Run this cleanup **before** the L3 #1 (`1_setup_resources.ipynb`) cleanup so the registry record drains before the registry itself is deleted.

Uncomment and run to delete all resources created in this notebook.

In [ ]:
## === CLEANUP: Delete all resources created in this notebook ===
## ⚠️ Uncomment and run this cell only after you are done with ALL notebooks.
## NOTE: Run this BEFORE the L3 #1 (1_setup_resources.ipynb) cleanup, so the
##       registry record drains before the registry itself is deleted.

# import boto3, os, time
#
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
#
# LAMBDA_NAME       = "anycompany_order_tools"
# LAMBDA_ROLE_NAME  = "AnyCompanyOrderToolsLambdaRole"
# RUNTIME_ROLE_NAME = "AgentCoreOrderAgentA2ARole"
# TARGET_NAME       = "order-tools"
# RUNTIME_NAME      = "order_agent_a2a"
# RECORD_NAME       = "order_agent_a2a_record"
# ECR_REPO_NAME     = "bedrock-agentcore-order_agent_a2a"
# CODEBUILD_PROJECT = "bedrock-agentcore-order_agent_a2a-builder"
#
# iam               = boto3.client("iam")
# lambda_client     = boto3.client("lambda", region_name=REGION)
# ssm               = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
# logs_client       = boto3.client("logs", region_name=REGION)
# ecr               = boto3.client("ecr", region_name=REGION)
# codebuild         = boto3.client("codebuild", region_name=REGION)
#
# def _ssm_get(key):
#     try:
#         return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound:
#         return None
#
# GATEWAY_ID  = _ssm_get("gateway_id")
# REGISTRY_ID = _ssm_get("registry_id")
#
# # ── Phase 1: Delete Registry Record (must precede L3#1 registry delete) ─────
# if REGISTRY_ID:
#     try:
#         records = agentcore_control.list_registry_records(registryId=REGISTRY_ID).get("registryRecords", [])
#         for rec in records:
#             if rec.get("name") == RECORD_NAME:
#                 try:
#                     agentcore_control.delete_registry_record(registryId=REGISTRY_ID, recordId=rec["recordId"])
#                     print(f"Deleted registry record: {RECORD_NAME}")
#                 except agentcore_control.exceptions.ResourceNotFoundException:
#                     print(f"Registry record {RECORD_NAME} already deleted.")
#                 except Exception as e:
#                     print(f"Registry record delete: {e}")
#                 break
#         else:
#             print(f"Registry record {RECORD_NAME} not found (already deleted).")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Registry already deleted; skipping record cleanup.")
#     except Exception as e:
#         print(f"List registry records: {e}")
#
# # ── Phase 2: Delete AgentCore Runtime (control plane API) and poll ──────────
# agent_runtime_id, agent_runtime_arn = None, None
# try:
#     paginator = agentcore_control.get_paginator("list_agent_runtimes")
#     for page in paginator.paginate():
#         for rt in page.get("agentRuntimes", []):
#             if rt.get("agentRuntimeName") == RUNTIME_NAME:
#                 agent_runtime_id  = rt.get("agentRuntimeId")
#                 agent_runtime_arn = rt.get("agentRuntimeArn")
#                 break
#         if agent_runtime_id:
#             break
# except Exception as e:
#     print(f"List agent runtimes: {e}")
#
# if agent_runtime_id:
#     try:
#         agentcore_control.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
#         print(f"Initiated delete of agent runtime: {agent_runtime_id}")
#         deadline = time.time() + 300  # 5 min
#         while time.time() < deadline:
#             try:
#                 agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id)
#                 time.sleep(10)
#             except agentcore_control.exceptions.ResourceNotFoundException:
#                 print("Agent runtime fully deleted.")
#                 break
#         else:
#             print("Timed out waiting for runtime to delete; continuing.")
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Agent runtime already deleted.")
#     except Exception as e:
#         print(f"Agent runtime delete: {e}")
# else:
#     print(f"Agent runtime {RUNTIME_NAME} not found (already deleted).")
#
# # ── Phase 3: Delete CloudWatch log delivery components ──────────────────────
# # Order: deliveries → destinations → sources (CloudWatch enforces this order)
# sources_for_runtime = []
# if agent_runtime_arn:
#     try:
#         paginator = logs_client.get_paginator("describe_delivery_sources")
#         for page in paginator.paginate():
#             for s in page.get("deliverySources", []):
#                 if agent_runtime_arn in s.get("resourceArns", []):
#                     sources_for_runtime.append(s)
#     except Exception as e:
#         print(f"List delivery sources: {e}")
#
# source_names = {s["name"] for s in sources_for_runtime}
# dest_arns_to_delete = set()
#
# if source_names:
#     try:
#         paginator = logs_client.get_paginator("describe_deliveries")
#         for page in paginator.paginate():
#             for d in page.get("deliveries", []):
#                 if d.get("deliverySourceName") in source_names:
#                     if d.get("deliveryDestinationArn"):
#                         dest_arns_to_delete.add(d["deliveryDestinationArn"])
#                     try:
#                         logs_client.delete_delivery(id=d["id"])
#                         print(f"Deleted delivery: {d['id']}")
#                     except Exception as e:
#                         print(f"Delete delivery ({d['id']}): {e}")
#     except Exception as e:
#         print(f"List deliveries: {e}")
#
# for dest_arn in dest_arns_to_delete:
#     dest_name = dest_arn.split(":delivery-destination:", 1)[-1] if ":delivery-destination:" in dest_arn else None
#     if dest_name:
#         try:
#             logs_client.delete_delivery_destination(name=dest_name)
#             print(f"Deleted delivery destination: {dest_name}")
#         except Exception as e:
#             print(f"Delete delivery destination ({dest_name}): {e}")
#
# for s in sources_for_runtime:
#     try:
#         logs_client.delete_delivery_source(name=s["name"])
#         print(f"Deleted delivery source: {s['name']}")
#     except Exception as e:
#         print(f"Delete delivery source ({s['name']}): {e}")
#
# # ── Phase 4: Delete CloudWatch log groups ───────────────────────────────────
# log_groups = [f"/aws/lambda/{LAMBDA_NAME}"]
# if agent_runtime_id:
#     log_groups += [
#         f"/aws/vendedlogs/bedrock-agentcore/runtime/APPLICATION_LOGS/{agent_runtime_id}",
#         f"/aws/vendedlogs/bedrock-agentcore/runtime/USAGE_LOGS/{agent_runtime_id}",
#     ]
# for lg in log_groups:
#     try:
#         logs_client.delete_log_group(logGroupName=lg)
#         print(f"Deleted log group: {lg}")
#     except logs_client.exceptions.ResourceNotFoundException:
#         pass
#     except Exception as e:
#         print(f"Log group delete ({lg}): {e}")
#
# # ── Phase 5: Delete Gateway Target ──────────────────────────────────────────
# if GATEWAY_ID:
#     try:
#         targets = agentcore_control.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
#         for t in targets.get("items", targets.get("targets", [])):
#             if t.get("name") == TARGET_NAME:
#                 try:
#                     agentcore_control.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=t["targetId"])
#                     print(f"Deleted gateway target: {TARGET_NAME}")
#                 except Exception as e:
#                     print(f"Gateway target delete: {e}")
#                 break
#     except agentcore_control.exceptions.ResourceNotFoundException:
#         print("Gateway already deleted; target removed implicitly.")
#     except Exception as e:
#         print(f"List gateway targets: {e}")
#
# # ── Phase 6: Delete Lambda function ─────────────────────────────────────────
# try:
#     lambda_client.delete_function(FunctionName=LAMBDA_NAME)
#     print(f"Deleted Lambda: {LAMBDA_NAME}")
# except lambda_client.exceptions.ResourceNotFoundException:
#     print(f"Lambda {LAMBDA_NAME} already deleted.")
# except Exception as e:
#     print(f"Lambda delete: {e}")
#
# # ── Phase 7: Delete IAM roles (Lambda + Runtime execution) ──────────────────
# for role_name in [LAMBDA_ROLE_NAME, RUNTIME_ROLE_NAME]:
#     try:
#         for ap in iam.list_attached_role_policies(RoleName=role_name).get("AttachedPolicies", []):
#             iam.detach_role_policy(RoleName=role_name, PolicyArn=ap["PolicyArn"])
#         for p in iam.list_role_policies(RoleName=role_name).get("PolicyNames", []):
#             iam.delete_role_policy(RoleName=role_name, PolicyName=p)
#         iam.delete_role(RoleName=role_name)
#         print(f"Deleted IAM role: {role_name}")
#     except iam.exceptions.NoSuchEntityException:
#         print(f"IAM role {role_name} already deleted.")
#     except Exception as e:
#         print(f"IAM role cleanup ({role_name}): {e}")
#
# # ── Phase 8: Delete ECR repository (force=True removes container images) ────
# try:
#     ecr.delete_repository(repositoryName=ECR_REPO_NAME, force=True)
#     print(f"Deleted ECR repository: {ECR_REPO_NAME}")
# except ecr.exceptions.RepositoryNotFoundException:
#     print(f"ECR repository {ECR_REPO_NAME} already deleted.")
# except Exception as e:
#     print(f"ECR delete: {e}")
#
# # ── Phase 9: Delete CodeBuild project (per-agent build project) ─────────────
# try:
#     codebuild.delete_project(name=CODEBUILD_PROJECT)
#     print(f"Deleted CodeBuild project: {CODEBUILD_PROJECT}")
# except Exception as e:
#     msg = str(e)
#     if "ResourceNotFoundException" in msg or "does not exist" in msg.lower():
#         print(f"CodeBuild project {CODEBUILD_PROJECT} already deleted.")
#     else:
#         print(f"CodeBuild delete: {e}")
#
# # ── Phase 10: Delete SSM parameters ─────────────────────────────────────────
# for param in ["order_lambda_arn", "order_gateway_target_id"]:
#     try:
#         ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}")
#         print(f"Deleted SSM parameter: {SSM_PREFIX}/{param}")
#     except ssm.exceptions.ParameterNotFound:
#         pass
#     except Exception as e:
#         print(f"SSM delete ({param}): {e}")
#
# print("\n✅ All Order Agent resources cleaned up.")